# Spurious Currents — Static Drop Verification

Stationary circular drop test with CSF surface tension (post sign-fix, commit `637fc27`).

**Setup:** R=0.2 in 1×1 box, ρ=300, μ=0.1, ε=0.01 (interface width), 40×40 SEM (p=7).  
Four surface tension values: σ ∈ {0.05, 0.1, 0.5, 1.0}, corresponding to La ∈ {600, 1200, 6000, 12000}.

**Key metric:** Spurious capillary number $Ca^* = \mu u_{\max}/\sigma$ — should plateau near machine precision for a correct CSF implementation.  
**Non-dimensional time:** $t^* = \sigma t/(\mu D)$ — collapses all σ cases to the same timescale.

Data: `results/fixed_sigma_sweep/sigma_*/ekin.csv`  
CSV columns: `t, Ekin, enst, u_max, kappa_max, kappa_min, kappa_rms, Fst_max, phi_min, phi_max`

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
import matplotlib.gridspec as gridspec
from mpi4py import MPI

from pysemtools.io.ppymech.neksuite import preadnek
from pysemtools.datatypes.msh import Mesh as msh_c
from pysemtools.datatypes.field import Field as field_c

# Physics parameters
mu  = 0.1
rho = 300.0
D   = 0.4
R   = 0.2

# Data root
RESULTS_DIR = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'results')
FIXED_SWEEP = os.path.join(RESULTS_DIR, 'fixed_sigma_sweep')

# CSV column indices
COL = dict(t=0, Ekin=1, enst=2, u_max=3,
           kappa_max=4, kappa_min=5, kappa_rms=6,
           Fst_max=7, phi_min=8, phi_max=9)

sigma_values = [1.0, 0.5, 0.1, 0.05]
colors       = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']   # one per sigma

def load(sig):
    """Load ekin.csv for a given sigma, return data dict.
    Skips t=0 row: at t=0 velocity is zero, which clips to 1e-20 on semilogy
    and distorts the axis range by many decades."""
    path = os.path.join(FIXED_SWEEP, f'sigma_{sig}', 'ekin.csv')
    if not os.path.exists(path):
        return None
    d      = np.genfromtxt(path, delimiter=',', comments='#')
    d      = d[d[:, COL['t']] > 0]    # skip t=0 (u=0 → Ca*=0 distorts semilogy)
    t      = d[:, COL['t']]
    t_star = sig * t / (mu * D)
    La     = rho * sig * D / mu**2
    return dict(sig=sig, La=La, t=t, t_star=t_star, d=d)

datasets = {sig: load(sig) for sig in sigma_values}
for sig, ds in datasets.items():
    if ds is not None:
        print(f'sigma={sig}  La={ds["La"]:.0f}  rows={len(ds["t"])}  t*_max={ds["t_star"][-1]:.0f}')
    else:
        print(f'sigma={sig}  --- MISSING ---')

## 1  Spurious capillary number vs non-dimensional time

$Ca^* = \mu u_{\max}/\sigma$ vs $t^* = \sigma t/(\mu D)$.

Plotting in $t^*$ collapses all σ cases onto the same timescale.  
A correct CSF implementation should show $Ca^*$ plateauing near machine precision (~10⁻⁸), **independent of σ and La**.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5), dpi=150)

for (sig, ds), col in zip(datasets.items(), colors):
    if ds is None: continue
    Ca = mu * ds['d'][:, COL['u_max']] / sig
    ax.semilogy(ds['t_star'], np.maximum(Ca, 1e-20),
                label=f'σ={sig}, La={ds["La"]:.0f}', color=col, lw=1.5)

ax.set_xlabel(r'$t^* = \sigma t\,/\,(\mu D)$')
ax.set_ylabel(r'$Ca^* = \mu\, u_{\max}\,/\,\sigma$')
ax.set_title('Spurious capillary number vs non-dimensional time\n'
             '(stationary drop, fixed CSF sign, ε=0.01, 40×40 SEM p=7)')
ax.legend()
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('Ca_star_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

## 2  Steady-state Ca* vs Laplace number

Extract the plateau $Ca^*_\infty$ = mean of $Ca^*$ over the final 20% of available $t^*$
and plot vs La on a log-log scale.

A **flat line** means the spurious currents are independent of surface tension strength → the implementation is correct across all La.  
A **positive slope** would indicate the algorithm breaks down at high La.

In [ ]:
La_vals, Ca_inf, sigma_labels = [], [], []

print(f'{"σ":>6}  {"La":>8}  {"t*_max":>8}  {"Ca*_∞":>12}')
print('-' * 42)
for sig, ds in datasets.items():
    if ds is None: continue
    Ca     = mu * ds['d'][:, COL['u_max']] / sig
    mask   = ds['t_star'] >= ds['t_star'][-1] * 0.8
    ca_p   = np.mean(Ca[mask])
    La_vals.append(ds['La'])
    Ca_inf.append(ca_p)
    sigma_labels.append(f'σ={sig}')
    print(f'{sig:6.2f}  {ds["La"]:8.0f}  {ds["t_star"][-1]:8.0f}  {ca_p:12.3e}')

fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
ax.loglog(La_vals, Ca_inf, 'o-', markersize=9, lw=2)
for La, Ca, lbl in zip(La_vals, Ca_inf, sigma_labels):
    ax.annotate(lbl, (La, Ca), textcoords='offset points',
                xytext=(6, 0), fontsize=9)

# Reference: flat line at the mean
ax.axhline(np.mean(Ca_inf), color='gray', ls='--', lw=1,
           label=f'mean = {np.mean(Ca_inf):.2e}')

ax.set_xlabel(r'$La = \rho\sigma D\,/\,\mu^2$')
ax.set_ylabel(r'$Ca^*_\infty$ (plateau, last 20% of $t^*$)')
ax.set_title('Steady-state spurious Ca* vs Laplace number')
ax.legend()
ax.grid(True, which='both', alpha=0.4)
plt.tight_layout()
plt.savefig('Ca_inf_vs_La.png', dpi=150, bbox_inches='tight')
plt.show()

## 3  Curvature quality

$\kappa_{\rm rms}$ (interface-weighted RMS) should stay close to the analytical value $1/R = 5.0$ throughout. Drift indicates systematic error in the curvature computation; oscillations indicate numerical noise in the phase field gradient.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5), dpi=150)

for (sig, ds), col in zip(datasets.items(), colors):
    if ds is None: continue
    ax.plot(ds['t_star'], ds['d'][:, COL['kappa_rms']],
            label=f'σ={sig}, La={ds["La"]:.0f}', color=col, lw=0.9)

ax.axhline(1/R, color='k', ls='--', lw=1.5, label=r'analytical $\kappa = 1/R = 5.0$')
ax.set_xlabel(r'$t^*$')
ax.set_ylabel(r'$\kappa_{\rm rms}$ (interface-weighted)')
ax.set_title('Curvature RMS vs non-dimensional time')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('kappa_rms_sigma_sweep.png', dpi=150, bbox_inches='tight')
plt.show()

# Late-time drift
print(f'{"σ":>6}  {"κ_rms (late)".rjust(14)}  {"drift from 1/R".rjust(16)}')
for sig, ds in datasets.items():
    if ds is None: continue
    k_late = ds['d'][:, COL['kappa_rms']][-int(0.2*len(ds['t'])):].mean()
    drift  = 100 * (k_late - 1/R) / (1/R)
    print(f'{sig:6.2f}  {k_late:14.4f}  {drift:+14.2f}%')

## 4  Phase field quality and surface tension force

**φ bounds:** φ should stay in [0, 1]. Small undershoots at machine precision are fine;
significant values indicate advection problems or insufficient interface resolution.

**F_ST max:** The maximum surface tension force magnitude should remain bounded.
Growth over time would indicate a force amplification instability.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), dpi=150)

# Panel 1: phi bounds
ax = axes[0]
for (sig, ds), col in zip(datasets.items(), colors):
    if ds is None: continue
    ax.plot(ds['t_star'], ds['d'][:, COL['phi_min']],
            color=col, lw=0.8, label=f'σ={sig}')
    ax.plot(ds['t_star'], ds['d'][:, COL['phi_max']],
            color=col, lw=0.8, ls='--')
ax.axhline(0, color='k', lw=0.8, alpha=0.5, ls='-')
ax.axhline(1, color='k', lw=0.8, alpha=0.5, ls='--')
ax.set_xlabel(r'$t^*$')
ax.set_ylabel(r'$\phi_{\min}$ (solid), $\phi_{\max}$ (dashed)')
ax.set_title('Phase field bounds (should stay in [0, 1])')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)

# Panel 2: Fst_max
ax = axes[1]
for (sig, ds), col in zip(datasets.items(), colors):
    if ds is None: continue
    ax.semilogy(ds['t_star'], ds['d'][:, COL['Fst_max']],
                color=col, lw=0.9, label=f'σ={sig}')
ax.set_xlabel(r'$t^*$')
ax.set_ylabel(r'$F_{\rm ST,max}$')
ax.set_title('Max surface tension force magnitude')
ax.legend(fontsize=8)
ax.grid(True, which='both', alpha=0.4)

plt.tight_layout()
plt.savefig('phi_Fst_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 5  Full diagnostics — σ = 1 (La = 12000)

Six-panel time series for the most demanding case (highest La).

In [ ]:
ds1 = datasets[1.0]
d   = ds1['d']
ts  = ds1['t_star']

Ca1 = mu * d[:, COL['u_max']] / 1.0

fig, axes = plt.subplots(2, 3, figsize=(15, 8), dpi=150)
axes = axes.ravel()

panels = [
    (ts, np.maximum(Ca1, 1e-20),                'semilogy', r'$Ca^*$',          r'$Ca^* = \mu u_{\max}/\sigma$'),
    (ts, np.maximum(d[:, COL['Ekin']], 1e-20),  'semilogy', r'$E_{\rm kin}$',   'Kinetic energy'),
    (ts, d[:, COL['kappa_rms']],                'plot',     r'$\kappa_{\rm rms}$','Curvature RMS'),
    (ts, d[:, COL['Fst_max']],                  'semilogy', r'$F_{\rm ST,max}$', 'Max ST force'),
    (ts, d[:, COL['phi_min']],                  'plot',     r'$\phi_{\min}$',    r'Phase field $\phi_{\min}$'),
    (ts, d[:, COL['phi_max']],                  'plot',     r'$\phi_{\max}$',    r'Phase field $\phi_{\max}$'),
]

for ax, (x, y, plot_fn, ylabel, title) in zip(axes, panels):
    getattr(ax, plot_fn)(x, y, lw=1.2)
    if title == r'Curvature RMS':
        ax.axhline(1/R, color='r', ls='--', lw=1, label='1/R = 5.0')
        ax.legend(fontsize=8)
    ax.set_xlabel(r'$t^*$'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.grid(True, which='both', alpha=0.4)

fig.suptitle(r'Full diagnostics — $\sigma=1$, La=12000, 40×40 SEM (p=7)',
             fontsize=12)
plt.tight_layout()
plt.savefig('diagnostics_sigma1.png', dpi=150, bbox_inches='tight')
plt.show()

## 6  Field visualization — σ = 1, final snapshot (t* ≈ 250)

Phase field φ, pressure p, and velocity magnitude |u| at t≈10 (t*≈250) for σ=1.

**What to look for:**
- φ: clean, sharp circular interface at r = R = 0.2 (centred at (0.5, 0.5))
- p: Laplace jump Δp = σ/R = 5.0 across the interface
- |u|: magnitude at machine precision (~μ⁻¹ Ca* σ ≈ 2×10⁻⁷) — effectively zero flow

Requires `field0.f00000` and `field0.f00100` in `results/fixed_sigma_sweep/sigma_1.0/fields/`.

In [ ]:
comm       = MPI.COMM_WORLD
FIELDS_DIR = os.path.join(FIXED_SWEEP, 'sigma_1.0', 'fields')

xyz_info = preadnek(os.path.join(FIELDS_DIR, 'field0.f00000'), comm)
msh      = msh_c(comm, data=xyz_info)

n_elem = msh.x.shape[0]
lx     = msh.x.shape[3]
ly     = msh.x.shape[2]

x_all, y_all, triangles = [], [], []
offset = 0
for e in range(n_elem):
    x_all.append(msh.x[e, 0, :, :].flatten())
    y_all.append(msh.y[e, 0, :, :].flatten())
    for j in range(ly - 1):
        for i in range(lx - 1):
            p0 = offset + j*lx + i
            p1 = offset + j*lx + (i+1)
            p2 = offset + (j+1)*lx + i
            p3 = offset + (j+1)*lx + (i+1)
            triangles.extend([[p0, p1, p3], [p0, p3, p2]])
    offset += lx * ly

x_f    = np.concatenate(x_all)
y_f    = np.concatenate(y_all)
triang = tri.Triangulation(x_f, y_f, np.array(triangles))
print(f'{n_elem} elements, {len(x_f)} points')

In [ ]:
data_t = preadnek(os.path.join(FIELDS_DIR, 'field0.f00100'), comm)
fld    = field_c(comm, data=data_t)
t_snap = fld.t

phi  = fld.fields['scal'][0][:, 0, :, :].flatten()
p    = fld.fields['pres'][0][:, 0, :, :].flatten()
u    = fld.fields['vel'][0][:, 0, :, :].flatten()
v    = fld.fields['vel'][1][:, 0, :, :].flatten()
umag = np.sqrt(u**2 + v**2)

dp_measured  = p[phi > 0.9].mean() - p[phi < 0.1].mean()
dp_theory    = 1.0 / R
Ca_star_snap = mu * umag.max() / 1.0
print(f't = {t_snap:.2f}  Δp = {dp_measured:.4f}  (theory {dp_theory:.4f})  '
      f'error = {100*abs(dp_measured-dp_theory)/dp_theory:.2f}%')
print(f'Ca* = {Ca_star_snap:.3e}')

theta = np.linspace(0, 2*np.pi, 300)

# Zoom to the drop region
pad  = 0.12
xlim = [0.5 - R - pad, 0.5 + R + pad]
ylim = [0.5 - R - pad, 0.5 + R + pad]

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5), dpi=150)

# Phase field
ax = axes[0]
cf = ax.tricontourf(triang, phi, levels=100, cmap='RdBu_r', vmin=0, vmax=1)
ax.tricontour(triang, phi, levels=[0.5], colors='k', linewidths=1.2)
ax.plot(0.5 + R*np.cos(theta), 0.5 + R*np.sin(theta),
        'k--', lw=1, label='analytical r=R')
ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.set_aspect('equal')
ax.set_title(r'Phase field $\phi$')
ax.set_xlabel('x'); ax.set_ylabel('y')
ax.legend(fontsize=8)
fig.colorbar(cf, ax=ax, label=r'$\phi$')

# Pressure
ax = axes[1]
cf2 = ax.tricontourf(triang, p, levels=100, cmap='seismic')
ax.tricontour(triang, phi, levels=[0.5], colors='k', linewidths=0.8, alpha=0.6)
ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.set_aspect('equal')
ax.set_title(fr'Pressure  ($\Delta p = {dp_measured:.3f}$, theory = {dp_theory:.3f})')
ax.set_xlabel('x'); ax.set_ylabel('y')
fig.colorbar(cf2, ax=ax, label='p')

# Velocity magnitude
ax = axes[2]
cf3 = ax.tricontourf(triang, umag, levels=100, cmap='hot_r')
ax.tricontour(triang, phi, levels=[0.5], colors='w', linewidths=0.8)
ax.set_xlim(xlim); ax.set_ylim(ylim)
ax.set_aspect('equal')
ax.set_title(fr'$|\mathbf{{u}}|$  ($Ca^* = {Ca_star_snap:.2e}$)')
ax.set_xlabel('x'); ax.set_ylabel('y')
fig.colorbar(cf3, ax=ax, label=r'$|\mathbf{u}|$')

fig.suptitle(fr'Static drop: $\sigma=1$, $t={t_snap:.1f}$ ($t^*\approx250$), 40×40 SEM (p=7)',
             fontsize=11)
plt.tight_layout()
plt.savefig('field_visualization_sigma1.png', dpi=150, bbox_inches='tight')
plt.show()

## 7  Pressure cross-section

Profile of pressure along y ≈ 0.5 through the drop centre.
The step should equal σ/R = 5.0.

In [ ]:
# Collect GLL points near the centreline y ≈ 0.5
tol  = 0.012
mask = np.abs(y_f - 0.5) < tol
xs_l = x_f[mask]
ps_l = p[mask]
idx  = np.argsort(xs_l)

# Measure jump: average inside (x ∈ [0.3,0.5]) and outside (x < 0.3)
in_drop  = (xs_l > 0.32) & (xs_l < 0.5)
out_drop = xs_l < 0.30
p_in     = ps_l[in_drop].mean()
p_out    = ps_l[out_drop].mean()
dp_line  = p_in - p_out

fig, ax = plt.subplots(figsize=(8, 4), dpi=150)
ax.plot(xs_l[idx], ps_l[idx], 'b.', ms=3, alpha=0.6, label='GLL nodes near y=0.5')
ax.axvline(0.5 - R, color='r', ls='--', lw=1.2, label=f'interface x={0.5-R:.2f}, {0.5+R:.2f}')
ax.axvline(0.5 + R, color='r', ls='--', lw=1.2)
ax.text(0.52, p_in, fr'$\Delta p = {dp_line:.3f}$ (theory {dp_theory:.3f})',
        va='center', fontsize=9)
ax.set_xlabel('x')
ax.set_ylabel('p')
ax.set_title(r'Pressure profile at $y \approx 0.5$  —  Laplace jump $\Delta p = \sigma/R$')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('pressure_profile_sigma1.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Δp measured = {dp_line:.4f},  theory = {dp_theory:.4f},  '
      f'error = {100*abs(dp_line-dp_theory)/dp_theory:.2f}%')